# Benchmark des traducteurs arabe → français

Trois modèles traduisent les mêmes segments MGB-2. Le gagnant traduira le reste du corpus : ses sorties deviennent les sous-titres français d'entraînement de la cascade (S2) et du projecteur (S3).

Données : le jeu commun du groupe, `donnees/marque/5331FFF2-….marque.txt` (217 blocs `<<id | début-fin>>` en arabe, émission 5331FFF2). La traduction française de Paulo, avec les mêmes blocs, se pose dans le même dossier : elle est trouvée seule et active chrF++, BLEU et COMET contre référence sur tous les segments. Tout se règle dans `configs/benchmark_traduction.yaml` (vider `marque_ar` pour revenir au tirage dans `donnees/*.xml`).

| Critère | Comment | Rôle |
|---|---|---|
| Sorties inutilisables | vide, arabe restant, « Voici la traduction », pas du français | élimine au-delà de 5 % |
| Sens (humain, 1–5) | grille à l'aveugle, A/B/C mélangés par segment | **décide** |
| Fluidité (humain, 1–5), noms et chiffres ok/ko, préférence | même grille | départage, analyse d'erreurs |
| chrF++, BLEU, COMET | contre la traduction de Paulo (ou `fr_reference` de la grille) | départage, classement provisoire |
| COMETKiwi | qualité estimée sans référence | départage, classement provisoire |
| Longueur, chiffres, noms du lexique, caractères par seconde | contrôles automatiques | signaux à relire |
| Temps par segment | mesuré, projeté sur 50 000 segments | départage, budget GPU |

Si `notebooks/seamless_m4t.ipynb` a tourné, ses traductions (son → français, sans texte arabe) sont relues ici et entrent dans le tableau, la grille et la décision comme un quatrième modèle.

GPU nécessaire pour les traductions (Kaggle ou Colab T4). `MODE_ESSAI=1` remplace les modèles par des sorties factices pour vérifier la chaîne.

**Lancer sur Colab ou Kaggle.** Depuis la racine du dépôt, avec la traduction de Paulo dans `donnees/marque/` :

```
zip -r NLP.zip code configs notebooks donnees/marque donnees/noms -x '*/__pycache__/*'
```

- Kaggle : `NLP.zip` en *Add Input*, GPU T4, Internet activé, secret `HF_TOKEN` (*Add-ons > Secrets*, licence Aya acceptée). Ajoute aussi en entrée la sortie `resultats.zip` de `seamless_m4t` pour l'avoir comme quatrième modèle.
- Colab : exécution GPU T4, secret `HF_TOKEN` (icône clé), envoyer `NLP.zip` et `resultats.zip` de `seamless_m4t` à l'invite. La dernière cellule télécharge `resultats.zip`.

In [6]:
# Colab ou Kaggle : installe, retrouve le projet (NLP.zip), lit le secret HF_TOKEN.
# Kaggle : NLP.zip en « Add Input », GPU T4 et Internet activés, secret HF_TOKEN dans Add-ons > Secrets.
# Colab : envoie NLP.zip à l'invite, secret HF_TOKEN dans le panneau « clé ».
# resultats.zip (sortie de l'autre notebook) est relu s'il est envoyé ou ajouté en entrée.
import os
import shutil
import sys
import zipfile
from pathlib import Path

SUR_COLAB = "google.colab" in sys.modules
SUR_KAGGLE = Path("/kaggle/working").is_dir()
if SUR_COLAB or SUR_KAGGLE:
    !pip install -q transformers accelerate bitsandbytes sentencepiece sacrebleu py3langid pyyaml huggingface_hub
    # unbabel-comet à part : il impose numpy<2 et peut rétrograder des paquets de Colab.
    # S'il échoue, chrF++ et BLEU restent mesurés et COMET est marqué « non mesuré ».
    AVEC_COMET = True
    if AVEC_COMET:
        !pip install -q "unbabel-comet>=2.2.7" || echo "unbabel-comet non installé : COMET non mesuré"
        from importlib.metadata import version
        if "numpy" in sys.modules and sys.modules["numpy"].__version__ != version("numpy"):
            print("numpy a changé de version : Exécution > Redémarrer la session, puis relance tout "
                  "(les traductions déjà écrites sont relues).")
    travail = Path("/content") if SUR_COLAB else Path("/kaggle/working")
    entrees = Path("/content") if SUR_COLAB else Path("/kaggle/input")
    chercher = entrees.glob if SUR_COLAB else entrees.rglob  # pas de parcours de Google Drive

    def trouver_projet(dossier):
        return next((p.parents[2] for p in Path(dossier).rglob("code/traduction/benchmark_traduction.py")), None)

    # Le code est recopié à chaque lancement : un NLP.zip plus récent remplace l'ancien projet.
    # resultats/ n'est pas dans NLP.zip, donc les traductions déjà écrites restent.
    source = trouver_projet(entrees) if SUR_KAGGLE else None
    archives = sorted(chercher("NLP*.zip"), key=lambda p: p.stat().st_mtime)
    if source is not None:
        shutil.copytree(source, travail / "projet", dirs_exist_ok=True)  # /kaggle/input est en lecture seule
    elif archives:
        zipfile.ZipFile(archives[-1]).extractall(travail / "projet")
    elif trouver_projet(travail / "projet") is None:
        if SUR_COLAB:
            from google.colab import files
            files.upload()  # NLP.zip, et resultats.zip de l'autre notebook s'il existe
            archives = sorted(chercher("NLP*.zip"), key=lambda p: p.stat().st_mtime)
        if not archives:
            raise FileNotFoundError("NLP.zip introuvable : Kaggle → Add Input ; Colab → relance la cellule et envoie-le.")
        zipfile.ZipFile(archives[-1]).extractall(travail / "projet")
    projet = trouver_projet(travail / "projet")
    for archive in sorted(chercher("resultats*.zip")):
        if not (projet / "resultats").is_dir():
            zipfile.ZipFile(archive).extractall(projet)
            print("Résultats relus :", archive)

    if not os.environ.get("HF_TOKEN"):
        try:
            if SUR_COLAB:
                from google.colab import userdata
                os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
            else:
                from kaggle_secrets import UserSecretsClient
                os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception as exc:
            print("Secret HF_TOKEN absent : Aya, COMET et COMETKiwi seront refusés.", type(exc).__name__)
    os.chdir(projet / "notebooks")
    print("Projet :", projet)

In [7]:
import json
import os
import sys
from pathlib import Path

# True = sorties factices (chaîne seule, secondes). False = vrais modèles (GPU, minutes/heures).
#MODE_ESSAI = os.environ.get("MODE_ESSAI", "0") == "1"  # ou : 
MODE_ESSAI = True

racine = Path.cwd()
while not (racine / "code" / "traduction" / "benchmark_traduction.py").exists() and racine.parent != racine:
    racine = racine.parent
if not (racine / "code" / "traduction" / "benchmark_traduction.py").exists():
    raise FileNotFoundError("Lance le notebook depuis le dépôt NLP (le dossier qui contient code/).")
if str(racine / "code" / "traduction") not in sys.path:
    sys.path.insert(0, str(racine / "code" / "traduction"))

import importlib

import benchmark_traduction as bt

# relance de la cellule = code à jour, sans redémarrer le noyau
bt = importlib.reload(bt)

config = bt.charger_config()
sortie = racine / config["donnees"]["dossier_sortie"]
if MODE_ESSAI:
    sortie = sortie / "essai"
sortie.mkdir(parents=True, exist_ok=True)
candidats = config["candidats"]
print("Sortie :", sortie)
for c in candidats:
    print(f"  {c['nom']:16s} {c['modele_id']:36s} {c['licence']}")

Sortie : /Users/ftayma24/SKOOL/ING3/cours-nlp/NLP/resultats/traduction/5331FFF2/essai
  nllb-1.3b        facebook/nllb-200-distilled-1.3B     CC-BY-NC 4.0
  qwen2.5-7b       Qwen/Qwen2.5-7B-Instruct             Apache 2.0
  aya-expanse-8b   CohereLabs/aya-expanse-8b            CC-BY-NC 4.0 (licence à accepter sur Hugging Face)


In [8]:
sel = config["selection"]
segments, chemin_fr = bt.charger_jeu(config, sortie, racine)
nb_ref = sum(1 for s in segments if s.get("fr_reference"))
print(f"Jeu : {len(segments)} segments, {sum(s['duree_s'] for s in segments) / 60:.1f} min, "
      f"{len({s['episode'] for s in segments})} émission(s)")
print(f"Référence française : {chemin_fr or 'aucune'} ({nb_ref}/{len(segments)} segments)")
for s in segments[:3]:
    print(f"  {s['duree_s']:5.1f}s  {s['ar'][:90]}")
    if s.get("fr_reference"):
        print(f"         {s['fr_reference'][:90]}")

Jeu : 217 segments, 24.9 min, 1 émission(s)
Référence française : aucune (0/217 segments)
    9.0s  أهلا بكم، دعا الرئيس الفرنسي فرانسوا هولاند مواطنيه إلى الوحدة الوطنية التي اعتبرها سلاح ف
    6.0s  وكانت مصادر قضائية أكدت وقوع حوادث لإطلاق نار على عدة مساجد في فرنسا
    8.0s  منذ وقوع الهجوم على مقر صحيفة شارلي إيبدو نهار الأربعاء. نتوقف مع هذا الخبر لنناقشه في محو


In [9]:
class TraducteurFactice:
    def __init__(self, nom):
        self.nom = nom

    def traduire(self, lot):
        return [(f"Le présentateur parle de la situation ({self.nom}).", 0.01) for _ in lot]

    def fermer(self):
        pass


def fabrique(candidat):
    if MODE_ESSAI:
        return TraducteurFactice(candidat["nom"])
    print(f"Chargement de {candidat['modele_id']}")
    return bt.fabriquer_traducteur(candidat, config)


traductions = {}
for candidat in candidats:
    traductions[candidat["nom"]] = bt.traduire_candidat(candidat, segments, sortie, fabrique)
    temps = sum(l["temps_s"] for l in traductions[candidat["nom"]]) / len(segments)
    print(f"{candidat['nom']:16s} terminé, {temps:.2f} s par segment")

autres = bt.charger_traductions_supplementaires(sortie, segments, list(traductions))
for nom, lignes in autres.items():
    traductions[nom] = lignes
    print(f"{nom:16s} relu depuis traductions_{nom}.jsonl (autre notebook)")

for s in segments[:2]:
    print("\nAR :", s["ar"])
    for nom, lignes in traductions.items():
        print(f"  {nom:16s}", next(l["fr"] for l in lignes if l["id"] == s["id"]))

nllb-1.3b        terminé, 0.01 s par segment
qwen2.5-7b       terminé, 0.01 s par segment
aya-expanse-8b   terminé, 0.01 s par segment

AR : أهلا بكم، دعا الرئيس الفرنسي فرانسوا هولاند مواطنيه إلى الوحدة الوطنية التي اعتبرها سلاح فرنسا الوحيد في مواجهة التطرف،
  nllb-1.3b        Le présentateur parle de la situation (nllb-1.3b).
  qwen2.5-7b       Le présentateur parle de la situation (qwen2.5-7b).
  aya-expanse-8b   Le présentateur parle de la situation (aya-expanse-8b).

AR : وكانت مصادر قضائية أكدت وقوع حوادث لإطلاق نار على عدة مساجد في فرنسا
  nllb-1.3b        Le présentateur parle de la situation (nllb-1.3b).
  qwen2.5-7b       Le présentateur parle de la situation (qwen2.5-7b).
  aya-expanse-8b   Le présentateur parle de la situation (aya-expanse-8b).


In [10]:
automatique = bt.controles_automatiques(segments, traductions)
cometkiwi = {} if MODE_ESSAI else bt.scores_cometkiwi(segments, traductions)

print(f"{'modèle':16s} {'inutilisable':>12s} {'COMETKiwi':>10s} {'s/segment':>10s} {'car./s':>7s}  signaux")
for nom, r in automatique.items():
    print(f"{nom:16s} {r['echec_dur_pct']:>11.1f}% {str(cometkiwi.get(nom, '-')):>10s} "
          f"{r['temps_moyen_s']:>10.2f} {str(r['caracteres_par_s_median']):>7s}  {r['signaux_pct']}")
if cometkiwi.get("_motif"):
    print("COMETKiwi non mesuré :", cometkiwi["_motif"])

reference = bt.scores_avec_reference(segments, traductions, avec_comet=not MODE_ESSAI)
if reference:
    print(f"\nContre la traduction de référence ({next(iter(reference.values()))['nb']} segments) :")
    print(f"{'modèle':16s} {'chrF++':>7s} {'BLEU':>6s} {'COMET':>7s}")
    for nom, r in reference.items():
        print(f"{nom:16s} {r['chrf_pp']:>7} {r['bleu']:>6} {r['comet']:>7}")

for nom, lignes in traductions.items():
    bt.ecrire_jsonl(sortie / f"traductions_{nom}.jsonl", lignes)

modèle           inutilisable  COMETKiwi  s/segment  car./s  signaux
nllb-1.3b                0.0%          -       0.01     7.1  {'chiffre_manquant': 2.3, 'longueur_suspecte': 38.7, 'nom_propre_different': 17.1}
qwen2.5-7b               0.0%          -       0.01     7.3  {'chiffre_manquant': 2.8, 'longueur_suspecte': 35.9, 'nom_propre_different': 17.1}
aya-expanse-8b           0.0%          -       0.01     7.9  {'chiffre_manquant': 2.8, 'longueur_suspecte': 25.3, 'nom_propre_different': 17.1}


In [11]:
chemin_grille, chemin_cle = bt.ecrire_grille_aveugle(segments, traductions, sortie, graine=sel["graine"])
print("Grille à copier pour chaque annotateur :", chemin_grille)
print("Clé (ne pas ouvrir avant la fin des notes) :", chemin_cle)

Grille à copier pour chaque annotateur : /Users/ftayma24/SKOOL/ING3/cours-nlp/NLP/resultats/traduction/5331FFF2/essai/grille_aveugle_vierge.csv
Clé (ne pas ouvrir avant la fin des notes) : /Users/ftayma24/SKOOL/ING3/cours-nlp/NLP/resultats/traduction/5331FFF2/essai/cle_secrete.json


## Notation humaine

Chaque annotateur copie `grille_aveugle_vierge.csv` en `grille_<initiales>.csv` dans le même dossier, sans regarder `cle_secrete.json`. Deux annotateurs au moins, chacun sur les 60 lignes.

- `sens_A/B/C` : 5 tout le sens est rendu ; 4 nuance perdue ; 3 une information manque ou est fausse ; 2 sens en grande partie faux ; 1 contresens ou invention.
- `fluidite_A/B/C` : 5 français naturel de journal ; 1 illisible.
- `noms_chiffres_A/B/C` : `ok`, `ko`, ou `na` s'il n'y a ni nom ni chiffre.
- `prefere` : la lettre de la meilleure traduction, ou `egal`.
- `fr_reference` (facultatif, une seule personne suffit) : ta propre traduction. Vingt lignes remplies activent chrF++, BLEU et COMET.

Le `contexte_ar` aide à comprendre une phrase coupée ; on ne note que la traduction du segment.

In [12]:
grilles = sorted(p for p in sortie.glob("grille_*.csv") if p.name != "grille_aveugle_vierge.csv")
humain = None
if grilles:
    humain = bt.noter_grilles(grilles, chemin_cle, segments, traductions,
                              nb_tirages=config["decision"]["nb_tirages_bootstrap"], avec_comet=not MODE_ESSAI)
    print(f"{len(grilles)} grille(s). Accord sur la préférence : {humain['accord_preference_pct']} %")
    for nom, h in humain["humain"].items():
        print(f"  {nom:16s} {h}")
    for nom, r in humain["avec_reference"].items():
        print(f"  {nom:16s} référence : {r}")
else:
    print("Aucune grille notée : décision provisoire.")
if reference:
    humain = {**(humain or {}), "avec_reference": reference}

decision = bt.choisir_modele(
    automatique, humain, cometkiwi,
    echec_dur_max_pct=config["decision"]["echec_dur_max_pct"],
    nb_segments_cible=config["decision"]["nb_segments_cible_corpus"],
)
print(json.dumps(decision, ensure_ascii=False, indent=2))
(sortie / "decision.json").write_text(json.dumps(
    {"decision": decision, "automatique": automatique, "cometkiwi": cometkiwi, "humain": humain},
    ensure_ascii=False, indent=2), encoding="utf-8")

if not MODE_ESSAI:
    bt.enregistrer_dans_resultats(automatique, humain, cometkiwi, candidats, jeu=config["donnees"]["jeu"])
    print("Lignes ajoutées à resultats/resultats.csv")

Aucune grille notée : décision provisoire.
{
  "choix": "nllb-1.3b",
  "provisoire": true,
  "a_egalite_avec": [
    "qwen2.5-7b",
    "aya-expanse-8b"
  ],
  "ecartes": {},
  "heures_projetees_corpus": 0.1,
  "raison": "aucune note humaine : classement par COMETKiwi puis vitesse"
}


In [13]:
# Colab efface tout à la fin de la session ; Kaggle garde /kaggle/working comme sortie du notebook.
# resultats.zip se donne à l'autre notebook (envoi Colab, ou « Add Input » de cette sortie sur Kaggle).
if SUR_COLAB or SUR_KAGGLE:
    archive = shutil.make_archive(str(travail / "resultats"), "zip", root_dir=racine, base_dir="resultats")
    print("Résultats :", archive)
    if SUR_COLAB:
        from google.colab import files
        files.download(archive)